# 16. Data Association과 Unknown Correspondence

SLAM에서 센서가 landmark를 보더라도, 그 관측이 **어느 landmark에 대응되는지** 모를 수 있다.

$$c_t^i \in \{1,\dots,N,\text{new}\}$$

Data association은 관측 $z_t^i$와 지도 landmark $m_j$ 사이의 correspondence를 추정하는 문제다. 잘못 연결하면 localization과 mapping이 동시에 망가진다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False


## 1. Mahalanobis Gating

관측 예측값과 실제 관측의 차이를 innovation이라 하고, uncertainty를 반영한 거리는 Mahalanobis distance로 계산한다.

In [ ]:
np.random.seed(16)
landmarks = np.array([[1.0, 1.0], [3.5, 1.4], [2.2, 3.2], [5.0, 3.6], [6.2, 1.0]])
robot = np.array([2.8, 2.0])
R = np.diag([0.25**2, 0.25**2])
Rinv = np.linalg.inv(R)

true_ids = np.array([1, 2, 4])
measurements = landmarks[true_ids] + np.random.multivariate_normal([0, 0], R, len(true_ids))
measurements = np.vstack([measurements, [4.2, 2.7]])

def mahalanobis(z, m):
    d = z - m
    return float(np.sqrt(d.T @ Rinv @ d))

D = np.array([[mahalanobis(z, m) for m in landmarks] for z in measurements])
gate = 2.5
assignments = []
for row in D:
    j = int(np.argmin(row))
    assignments.append(j if row[j] < gate else None)

print('distance matrix:')
print(np.round(D, 2))
print('assignments:', assignments)

## 2. Nearest Neighbor Association 시각화

가장 가까운 landmark를 고르는 방식은 단순하지만, landmark가 촘촘하거나 노이즈가 커지면 ambiguous association이 자주 생긴다.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(landmarks[:,0], landmarks[:,1], c='tab:blue', s=90, label='map landmarks')
ax.scatter(measurements[:,0], measurements[:,1], c='tab:red', marker='x', s=110, label='measurements')
ax.scatter(robot[0], robot[1], c='black', marker='^', s=110, label='robot')

for j, m in enumerate(landmarks):
    ax.text(m[0] + 0.06, m[1] + 0.06, f'm{j}')
for i, z in enumerate(measurements):
    ax.text(z[0] + 0.06, z[1] - 0.16, f'z{i}')
    if assignments[i] is not None:
        m = landmarks[assignments[i]]
        ax.plot([z[0], m[0]], [z[1], m[1]], 'k--', alpha=0.45)
    else:
        ax.add_patch(plt.Circle(z, gate * np.sqrt(R[0,0]), fill=False, color='tab:red', alpha=0.35))
        ax.text(z[0] + 0.08, z[1] + 0.18, 'new?')

ax.set_aspect('equal', adjustable='box')
ax.set_title('Nearest-neighbor data association')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('assets/16_data_association.png', dpi=160)
plt.show()

## 3. Compatibility Matrix

SLAM에서는 단일 nearest neighbor만 보지 않고, 여러 관측이 동시에 서로 모순되지 않는지도 확인한다.

In [ ]:
compatible = D < gate
fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(compatible, cmap='Greens', vmin=0, vmax=1)
ax.set_title('Compatibility matrix: $d_M < gate$')
ax.set_xlabel('landmark id')
ax.set_ylabel('measurement id')
ax.set_xticks(range(len(landmarks)))
ax.set_yticks(range(len(measurements)))
for i in range(compatible.shape[0]):
    for j in range(compatible.shape[1]):
        ax.text(j, i, f'{D[i,j]:.1f}', ha='center', va='center', color='black')
plt.tight_layout()
plt.savefig('assets/16_compatibility_matrix.png', dpi=160)
plt.show()

## 4. 로보틱스 연결

| 개념 | 의미 | 로보틱스 활용 |
|---|---|---|
| Mahalanobis distance | 불확실성을 반영한 innovation 거리 | EKF-SLAM landmark matching |
| Gating | 말이 안 되는 correspondence 제거 | outlier rejection |
| Unknown correspondence | 관측 ID가 없는 상황 | real-world feature SLAM |
| New landmark decision | 기존 landmark와 맞지 않음 | map augmentation |

Data association은 수식보다 실전에서 더 어렵다. perceptual aliasing이 있으면 같은 모양의 장소나 landmark가 잘못 연결될 수 있다.